In [2]:
import pandas as pd
import numpy as np
import rdkit.Chem as Chem

In [14]:
df = pd.read_csv("Rf&nat.csv")

In [15]:
eluents = pd.concat([df['eluent1_smiles'],df['eluent2_smiles']]).drop_duplicates()

In [16]:
eluents_smi = []
for smi in eluents:
    try:
        print(smi)
        can_smi =Chem.MolToSmiles(Chem.MolFromSmiles(smi))
    except:
        pass
    if can_smi:
        eluents_smi.append(can_smi)
eluents_smi=np.unique(eluents_smi)

CCCCCC
ClCCl
ClC(Cl)Cl
CCOC(C)=O
Cc1ccccc1
CO
CCCCC
CC(C)=O
CCOCC
C1CCCCC1
c1ccccc1
CC#N
CCCCCCC
CCO
CC(C)(C)O
O=C(OCC)C
COC(C)(C)C
ClCCCl
CCN(CC)CC
C1CCOC1
C[N+](=O)[O-]
CCC


In [17]:
from rdkit.Chem import Descriptors, Crippen, rdMolDescriptors

def solvent_properties(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    props = {"SMILES":smiles}
    
    # 기본 물성
    props["MolecularWeight"] = Descriptors.MolWt(mol)
    props["LogP"] = Crippen.MolLogP(mol)  # hydrophobicity
    props["MolMR"] = Crippen.MolMR(mol)   # molar refractivity

    # 극성 관련
    props["TPSA"] = rdMolDescriptors.CalcTPSA(mol)
    props["HBA"] = rdMolDescriptors.CalcNumHBA(mol)
    props["HBD"] = rdMolDescriptors.CalcNumHBD(mol)

    # 구조적
    props["RotatableBonds"] = rdMolDescriptors.CalcNumRotatableBonds(mol)
    props["AromaticProportion"] = rdMolDescriptors.CalcFractionCSP3(mol)

    return props

In [18]:
data= []
for smi in eluents_smi:
    data.append(solvent_properties(smi))

In [19]:
solv_feat = pd.read_csv("solvent_feature.csv")
print(solv_feat.columns)

Index(['SMILES', 'MolecularWeight', 'LogP', 'MolMR', 'TPSA', 'HBA', 'HBD',
       'RotatableBonds', 'AromaticProportion', 'dipole', 'dielec'],
      dtype='object')


In [27]:
df = pd.read_csv("Rf&nat.csv")

In [28]:
eluents_smi=[]
for smi in df['eluent1_smiles']:
    try:
        can_smi =Chem.MolToSmiles(Chem.MolFromSmiles(smi))
    except:
        eluents_smi.append(None)
        continue
    if can_smi:
        eluents_smi.append(can_smi)
    else:
        eluents_smi.append(None)
df['eluent1_smiles']=eluents_smi

eluents_smi=[]
for smi in df['eluent2_smiles']:
    try:
        can_smi =Chem.MolToSmiles(Chem.MolFromSmiles(smi))
    except:
        eluents_smi.append(None)
        continue
    if can_smi:
        eluents_smi.append(can_smi)
    else:
        eluents_smi.append(None)
df['eluent2_smiles']=eluents_smi
df = df[-(pd.isna(df['eluent1_smiles']) | pd.isna(df['eluent2_smiles']))]


In [29]:
elue1_feat = solv_feat.copy()
elue1_feat.columns = ['eluent1_smiles', 'MolecularWeight1', 'LogP1', 'MolMR1', 'TPSA1', 'HBA1', 'HBD1',
       'RotatableBonds1', 'AromaticProportion1', 'dipole1', 'dielec1']
elue2_feat = solv_feat.copy()
elue2_feat.columns = ['eluent2_smiles', 'MolecularWeight2', 'LogP2', 'MolMR2', 'TPSA2', 'HBA2', 'HBD2',
       'RotatableBonds2', 'AromaticProportion2', 'dipole2', 'dielec2']

In [30]:
df = pd.merge(df,elue1_feat,'left',on='eluent1_smiles',).merge(elue2_feat,'left',on='eluent2_smiles',)
df.to_csv("Rf&nat_sf.csv",index=False)

In [26]:
df

,Smiles,eluent1_smiles,eluent2_smiles,ratio1,ratio2,Rf,set,MW1,LogP1_x,MolMR1_x,...,MolecularWeight2_y,LogP2,MolMR2,TPSA2,HBA2,HBD2,RotatableBonds2_y,AromaticProportion2,dipole2,dielec2
0,C[C@H](O[Si](C)(C)C(C)(C)C)[C@@H](OCc1ccccc1)[...,CCCCCC,COC(C)(C)C,0.784562,0.215438,0.220000,train,86.178,2.5866,29.816,...,88.150,1.4313,26.7620,9.23,1,0,0,1.00,1.32,4.50
1,CCCCCCCC/C=C\[C@@H](CC(=O)OC(C)(C)C)NC(=O)OC(C...,CCCCCC,COC(C)(C)C,0.879277,0.120723,0.420000,train,86.178,2.5866,29.816,...,88.150,1.4313,26.7620,9.23,1,0,0,1.00,1.32,4.50
2,CCCCCCCC/C=C\[C@H](CC(=O)OC(C)(C)C)NC(=O)OC(C)...,CCCCCC,COC(C)(C)C,0.879277,0.120723,0.420000,train,86.178,2.5866,29.816,...,88.150,1.4313,26.7620,9.23,1,0,0,1.00,1.32,4.50
3,BrCc1ccc(-c2ccccc2)nc1,ClCCl,CO,0.926681,0.073319,0.500000,train,84.933,1.4215,16.573,...,32.042,-0.3915,8.1428,20.23,1,1,0,1.00,1.60,32.70
4,COC(=O)c1ccc(NC(CCP(=O)(c2ccccc2)c2ccccc2)P(=O...,ClCCl,CO,0.926681,0.073319,0.350000,train,84.933,1.4215,16.573,...,32.042,-0.3915,8.1428,20.23,1,1,0,1.00,1.60,32.70
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29378,O=C1CCCCCC1,ClCCl,CCCCCC,1.000000,0.000000,0.549300,test,NaN,NaN,NaN,...,86.178,2.5866,29.8160,0.00,0,0,3,1.00,0.00,1.90
29379,Oc1cc(Br)ccc1I,ClCCl,CO,0.990099,0.009901,0.775384,test,NaN,NaN,NaN,...,32.042,-0.3915,8.1428,20.23,1,1,0,1.00,1.60,32.70
29380,Cc1ccc(I)cc1,ClCCl,CO,0.952381,0.047619,1.000000,test,NaN,NaN,NaN,...,32.042,-0.3915,8.1428,20.23,1,1,0,1.00,1.60,32.70
29381,O=Cc1cc2c(cc1Br)OCO2,ClCCl,CCCCCC,1.000000,0.000000,0.684828,test,NaN,NaN,NaN,...,86.178,2.5866,29.8160,0.00,0,0,3,1.00,0.00,1.90


In [48]:
path = "data/only_nat_PE_hex_filtered.csv"
df = pd.read_csv(path)
df

,Smiles,eluent1_smiles,eluent2_smiles,ratio1,ratio2,Rf
0,O=C(CCCl)c1ccccc1,CCCCCC,CCCCCC,1.0,0.0,0.049175
1,O=C(CCCl)c1ccccc1,O=C(OCC)C,CCCCCC,1.0,0.0,0.980526
2,CC(=O)c1ccc(C(C)=O)cc1,CCCCCC,CCCCCC,1.0,0.0,0.013357
3,CC(=O)c1ccc(C(C)=O)cc1,O=C(OCC)C,CCCCCC,1.0,0.0,0.912367
4,O=C(c1ccccc1)c1ccccn1,CCCCCC,CCCCCC,1.0,0.0,0.015686
...,...,...,...,...,...,...
4714,CC1(C)C=CC(=O)CC1,CCCCCC,CCOCC,0.5,0.5,0.596871
4715,CCC(=O)c1nccs1,CCCCCC,CCOCC,0.5,0.5,0.612287
4716,O=C(CCc1ccccc1)c1ccccc1,CCCCCC,CCOCC,0.5,0.5,0.802801
4717,CC1CCC(=O)c2ccccc21,CCCCCC,CCOCC,0.5,0.5,0.662423


In [59]:
df = df.sample(frac=1).reset_index(drop=True)
df.loc[:int(len(df)*0.8),'set']='train'
df.loc[int(len(df)*0.8):int(len(df)*0.9),'set']='val'
df.loc[int(len(df)*0.9):,'set']='test'


In [61]:
df.to_csv(path,index=False)

In [60]:
df[df['set']=='val']

,Smiles,eluent1_smiles,eluent2_smiles,ratio1,ratio2,Rf,set
3775,CC(=O)Nc1ccccc1,CCCCCC,O=C(OCC)C,0.500000,0.500000,0.392548,val
3776,Oc1ccc2ccccc2c1-c1c(O)ccc2ccccc12,CCCCCC,O=C(OCC)C,0.980392,0.019608,0.031782,val
3777,CC(=O)c1ccncc1,CCCCCC,CCOCC,0.500000,0.500000,0.149247,val
3778,O=Cc1ccc(O)c([N+](=O)[O-])c1,CCCCCC,CCCCCC,1.000000,0.000000,0.009113,val
3779,OCc1ccc(Br)cc1,CCCCCC,O=C(OCC)C,0.750000,0.250000,0.319097,val
...,...,...,...,...,...,...,...
4242,CC(=O)Oc1ccc(C(C)=O)cc1,ClCCl,CO,0.967742,0.032258,0.544513,val
4243,C/C(=N\O)c1ccccc1,ClCCl,CO,0.952381,0.047619,0.578505,val
4244,Oc1ccc(F)cc1,CCCCCC,O=C(OCC)C,0.500000,0.500000,0.839927,val
4245,O=Cc1ccc2ccccc2c1Br,O=C(OCC)C,CCCCCC,1.000000,0.000000,0.968668,val
